# Experimental Codes

In [ ]:
from TableTennisEnvironmentV0.TableTennisEnvironment import TableTennisEnv
from utils.math_solvers import solve_quadratic
import pybullet as p 

g = 9.8
env = TableTennisEnv(show_gui=False)

In [ ]:
## We need to estimate the initial velocity of the ball 
## although we can get it in pybullet, we won't get it directly in optitrack

def estimateInitVelocity(ball_position1, ball_position2, time_gap):
    displacement = (ball_position2[0] - ball_position1[0], ball_position2[1] - ball_position1[1], ball_position2[2] - ball_position1[2])
    velocity = [displacement[i] / time_gap for i in range(len(displacement))]
    return velocity
    

In [ ]:
## This is real initial velocity
u_x, u_y, u_z = 5.5, 0.8, -3     # in meters/sec
real_initial_velocity = [u_x, u_y, u_z]
env.throw_ball(real_initial_velocity)

## now we estimate the initial velocity
ball_position1 = p.getBasePositionAndOrientation(env._ball)[0]
action = [0, 0, 0, 0, 0, 0]
env.step(action)
ball_position2 = p.getBasePositionAndOrientation(env._ball)[0]
time_gap = 1/240

estimated_initial_velocity = estimateInitVelocity(ball_position1, ball_position2, time_gap)

print('real velocity: ', real_initial_velocity)
print('estimated velocity: ', estimated_initial_velocity)



So the results are pretty accurate

![Alt text](illustration.jpeg)


In [ ]:
# dimensions of the table
[Lx, Ly, Lz] = [2.74, 1.5, 1.0]


# distance between ball and the robot
ball_initial_position = p.getBasePositionAndOrientation(env._ball)[0] # can be collected from optitrack
robot_initial_position = p.getLinkState(env._robotic_arm, 0)[0]      # can be collected from optitrack
distance_x, distance_y, distance_z = (robot_initial_position[0] - ball_initial_position[0], robot_initial_position[1] - ball_initial_position[1], robot_initial_position[2] - ball_initial_position[2])

# height of the ball from the table
h1 = ball_initial_position[2] - Lz - 0.2 # 0.2m is the height of the red surface


# now calculate the time for the contact with ground
g = 9.8
a = -0.5*g
b = estimated_initial_velocity[2]
c = h1

t1 = None # it is the time for the ball to hit the ground for the first time

roots = solve_quadratic(a, b, c)
for root in roots:
    if root.imag == 0 and root.real > 0: # means complex root, ignore
        t1 = root.real


## now how much distance did the 
d1_x = estimated_initial_velocity[0]*t1
d1_y = estimated_initial_velocity[1]*t1


## Velocities at this position
vx = estimated_initial_velocity[0]
vy = estimated_initial_velocity[1]
vz = estimated_initial_velocity[2] - g*t1


## remaining distance that needs to be covered to reach the robot - maybe we need to set constraints on y displacement as well
d2_x = distance_x - d1_x

new_velocity = [vx, vy, -vz] # vz will change sign because now the direction of the projectile in z axis is reversed


## not calculate how high the projectile can go 
t2 = distance_x / estimated_initial_velocity[0] - t1 # remaining flight time to reach the robot
h2 = new_velocity[2]*t2 - 0.5*g*(t2**2)


# now recompute the position of the ball on the Y-Z plane

y = ball_initial_position[1] + estimated_initial_velocity[1] * (t1 + t2)
z = h2 + Lz + 0.2

estimated_hitting_point = (y, z)
print('total flight time: ', t1+t2, 's')
print('The ball will hit the Y-Z plane at: ', estimated_hitting_point)

In [ ]:
def calculate_flight_times(ball_initial_position, robot_initial_position, estimated_initial_velocity, table_height): # this will return the two flight times - t1 and t2
    distance_x, distance_y, distance_z = (robot_initial_position[0] - ball_initial_position[0], robot_initial_position[1] - ball_initial_position[1], robot_initial_position[2] - ball_initial_position[2])

    # height of the ball from the table
    h1 = ball_initial_position[2] - table_height - 0.2 # 0.2m is the height of the red surface


    # now calculate the time for the contact with ground
    a = -0.5*g
    b = estimated_initial_velocity[2]
    c = h1

    t1 = None # it is the time for the ball to hit the ground for the first time

    roots = solve_quadratic(a, b, c)
    for root in roots:
        if root.imag == 0 and root.real > 0: # means complex root, ignore
            t1 = root.real
    ## not calculate how high the projectile can go 
    t2 = distance_x / estimated_initial_velocity[0] - t1 # remaining flight time to reach the robot

    return t1, t2


# print(calculate_flight_times(ball_initial_position, robot_initial_position, estimated_initial_velocity, Lz))



In [ ]:
def calculate_h2(estimated_initial_velocity, t1, t2): # this will provide the second height that the ball will reach: h2
    ## Velocities after bounce
    vx = estimated_initial_velocity[0]
    vy = estimated_initial_velocity[1]
    vz = estimated_initial_velocity[2] - g*t1
    new_velocity = [vx, vy, -vz] # vz will change sign because now the direction of the projectile in z axis is reversed
    h2 = new_velocity[2]*t2 - 0.5*g*(t2**2)
    return h2
    

In [ ]:
def estimate_hitting_point(ball_initial_position, robot_initial_position, estimated_initial_velocity, table_height, base_surface_thickness = 0.2): # this will return the estimated hitting point in the Y-Z plane
    t1, t2 = calculate_flight_times(ball_initial_position, robot_initial_position, estimated_initial_velocity, table_height)
    h2 = calculate_h2(estimated_initial_velocity, t1, t2)

    y = ball_initial_position[1] + estimated_initial_velocity[1] * (t1 + t2)
    z = h2 + table_height + base_surface_thickness
    return (y, z)
## get a new ball
env.get_new_ball(position=[-1, 0.1, 2])


## Throw the ball
u_x, u_y, u_z = 5.5, 0.8, -3     # in meters/sec
real_initial_velocity = [u_x, u_y, u_z]
env.throw_ball(real_initial_velocity)


ball_position1 = ball_initial_position = p.getBasePositionAndOrientation(env._ball)[0]
robot_initial_position = p.getLinkState(env._robotic_arm, 0)[0]      # can be collected from optitrack
action = [0, 0, 0, 0, 0, 0]
env.step(action)
ball_position2 = p.getBasePositionAndOrientation(env._ball)[0]

table_height = 1.0
estimated_initial_velocity = estimateInitVelocity(ball_position1, ball_position2, 1/240)
estimated_hitting_point = estimate_hitting_point(ball_initial_position, robot_initial_position, estimated_initial_velocity, table_height)
print(estimated_hitting_point)

In [ ]:
import time
def ball_crossed_yz(ball_current_position, robot_current_position):#this function will detect if the ball reached Y-Z plane
    ball_current_position = p.getBasePositionAndOrientation(env._ball)[0] # can be collected from optitrack
    x_ball, y_ball, z_ball = ball_current_position
    x_robot, y_robot, z_robot = robot_current_position

    if x_ball > x_robot:  # this condition ensures the ball crossed Y-Z plane
        return True

    else:
        return False


def convert_step_to_time(step_number, frequency = 240):
    return step_number/frequency

action = [0, 0, 0, 0, 0, 0] # keep the robot stopped, we don't need it.

for step in range(5000):
    env.step(action)
    ball_current_position = p.getBasePositionAndOrientation(env._ball)[0]
    if ball_crossed_yz(ball_current_position, robot_initial_position):
        # print(convert_step_to_time(step, 240))
        break
    time.sleep(1/240) # because the simulation is at 240 Hz

real_hitting_point = (p.getBasePositionAndOrientation(env._ball)[0][1], p.getBasePositionAndOrientation(env._ball)[0][2])
print('The ball has hit the Y-Z plane at: ', real_hitting_point)

# now calculate delta

In [ ]:
## calculate delta

delta = (real_hitting_point[0] - estimated_hitting_point[0], real_hitting_point[1] - estimated_hitting_point[1])
print('delta: ', delta)

# Simplified with utils/physics_solvers.py

In [1]:
from TableTennisEnvironmentV0.TableTennisEnvironment import TableTennisEnv
from utils.physics_solvers import estimateInitVelocity, estimate_hitting_point, calculate_delta
import time
import pybullet as p 

g = 9.8
env = TableTennisEnv(show_gui=False)

pybullet build time: Feb  4 2024 12:55:26


In [2]:
## Throw the ball
u_x, u_y, u_z = 5.5, 0.8, -3     # in meters/sec
real_initial_velocity = [u_x, u_y, u_z]
env.throw_ball(real_initial_velocity)


ball_position1 = ball_initial_position = p.getBasePositionAndOrientation(env._ball)[0]
robot_initial_position = p.getLinkState(env._robotic_arm, 0)[0]      # can be collected from optitrack
action = [0, 0, 0, 0, 0, 0]
env.step(action)
ball_position2 = p.getBasePositionAndOrientation(env._ball)[0]

table_height = 1.0
estimated_initial_velocity = estimateInitVelocity(ball_position1, ball_position2, 1/240)
estimated_hitting_point = estimate_hitting_point(ball_initial_position, robot_initial_position, estimated_initial_velocity, table_height)
print(estimated_hitting_point)

(0.39279742569090914, 2.2399837149277717)


In [3]:
def ball_crossed_yz(ball_current_position, robot_current_position):#this function will detect if the ball reached Y-Z plane
    ball_current_position = p.getBasePositionAndOrientation(env._ball)[0] # can be collected from optitrack
    x_ball, y_ball, z_ball = ball_current_position
    x_robot, y_robot, z_robot = robot_current_position

    if x_ball > x_robot:  # this condition ensures the ball crossed Y-Z plane
        return True

    else:
        return False

In [4]:
action = [0, 0, 0, 0, 0, 0] # keep the robot stopped, we don't need it.

for step in range(5000):
    env.step(action)
    ball_current_position = p.getBasePositionAndOrientation(env._ball)[0]
    if ball_crossed_yz(ball_current_position, robot_initial_position):
        # print(convert_step_to_time(step, 240))
        break
    time.sleep(1/240) # because the simulation is at 240 Hz

real_hitting_point = (p.getBasePositionAndOrientation(env._ball)[0][1], p.getBasePositionAndOrientation(env._ball)[0][2])
print('The ball has hit the Y-Z plane at: ', real_hitting_point)


The ball has hit the Y-Z plane at:  (0.3670424977504225, 1.4559086300728026)


In [5]:
print(calculate_delta(real_hitting_point, estimated_hitting_point))

(-0.025754927940486616, -0.7840750848549691)


In [6]:
## get a new ball for the next iteration
env.get_new_ball(position=[-1.5, -0.1, 1.8])